<a href="https://colab.research.google.com/github/mcarden4-netizen/Homework-Intro-to-ML/blob/Homework/Homework7part2b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%matplotlib inline

from matplotlib import pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import time
import copy

In [2]:
torch.manual_seed(123)
np.random.seed(123)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Using device:", device)

Using device: cpu


In [3]:
data = './data'

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer','dog', 'frog', 'horse', 'ship', 'truck']

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.4915, 0.4823, 0.4468),(0.2470, 0.2435, 0.2616))])

cifar10 = datasets.CIFAR10(data, train=True, download=True, transform=transform)

cifar10_val = datasets.CIFAR10(data, train=False, download=True, transform=transform)

100%|██████████| 170M/170M [23:41<00:00, 120kB/s]


In [4]:
Ts = len(cifar10)
Vs = int(0.1 * Ts)

cifar10t, cifar10v = random_split(cifar10, [Ts - Vs, Vs], generator=torch.Generator().manual_seed(50))

batch_size = 64

train_loader = DataLoader(cifar10t, batch_size=batch_size, shuffle=True)

train_eval_loader = DataLoader(cifar10t, batch_size=batch_size, shuffle=False)

val_loader = DataLoader(cifar10v, batch_size=batch_size, shuffle=False)

test_loader = DataLoader(cifar10_val, batch_size=batch_size, shuffle=False)

In [5]:
model4 = nn.Sequential(
    nn.Linear(3072, 1024),
    nn.ReLU(),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Linear(512, 10)
).to(device)

In [6]:
model5 = nn.Sequential(
    nn.Linear(3072, 512),
    nn.Sigmoid(),
    nn.Linear(512, 256),
    nn.Sigmoid(),
    nn.Linear(256, 10)
).to(device)

In [7]:
model6 = nn.Sequential(
    nn.Linear(3072, 512),
    nn.Tanh(),
    nn.Linear(512, 256),
    nn.Tanh(),
    nn.Linear(256, 10)
).to(device)

In [8]:
def evaluate(model, loader, loss_fn):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for imgs, labels in loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs.view(imgs.shape[0], -1))

            loss = loss_fn(outputs, labels)

            total_loss += loss.item() * imgs.shape[0]

            _, predicted = torch.max(outputs, dim=1)

            total += labels.shape[0]
            correct += (predicted == labels).sum().item()

    average_loss = total_loss / total
    accuracy = 100 * correct / total

    return average_loss, accuracy

In [9]:
def train_model(model, model_name):

    loss_fn = nn.CrossEntropyLoss()

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    n_epochs = 100
    patience = 50

    best_val_loss = float('inf')
    best_model = copy.deepcopy(model.state_dict())

    epochs_without_improvement = 0

    train_losses = []
    val_losses = []

    start_time = time.time()

    for epoch in range(n_epochs):

        model.train()

        total_loss = 0
        total = 0

        for imgs, labels in train_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs.view(imgs.shape[0], -1))

            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * imgs.shape[0]
            total += labels.shape[0]

        train_loss = total_loss / total

        val_loss, val_accuracy = evaluate(model, val_loader, loss_fn)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_model = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        if (epoch + 1) % 5 == 0:

            print(
                f'{model_name} | Epoch {epoch + 1}/{n_epochs} '
                f'| Train Loss: {train_loss:.4f} '
                f'| Validation Loss: {val_loss:.4f} '
                f'| Validation Accuracy: {val_accuracy:.2f}%'
            )

        if epochs_without_improvement >= patience:

            print(f'{model_name}: Early stopping at epoch {epoch + 1}')
            break

    training_time = time.time() - start_time

    model.load_state_dict(best_model)

    final_train_loss, train_accuracy = evaluate(model, train_eval_loader, loss_fn)

    final_val_loss, val_accuracy = evaluate(model, val_loader, loss_fn)

    test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)

    parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("\n", model_name, "Final Results")
    print("--------------------------------")
    print(f"Best Epoch: {best_epoch}")
    print(f"Training Time: {training_time:.2f} seconds")
    print(f"Training Loss: {final_train_loss:.4f}")
    print(f"Training Accuracy: {train_accuracy:.2f}%")
    print(f"Validation Accuracy: {val_accuracy:.2f}%")
    print(f"Test Accuracy: {test_accuracy:.2f}%")
    print(f"Trainable Parameters: {parameters:,}")

    results = {
        'name': model_name,
        'training_time': training_time,
        'train_loss': final_train_loss,
        'train_accuracy': train_accuracy,
        'val_accuracy': val_accuracy,
        'test_accuracy': test_accuracy,
        'parameters': parameters,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_epoch': best_epoch
    }

    return results

In [10]:
results4 = train_model(model4, "Experiment 1 - Increasing Width")

Experiment 1 - Increasing Width | Epoch 5/100 | Train Loss: 1.2397 | Validation Loss: 1.4292 | Validation Accuracy: 50.46%
Experiment 1 - Increasing Width | Epoch 10/100 | Train Loss: 0.9233 | Validation Loss: 1.6416 | Validation Accuracy: 51.62%
Experiment 1 - Increasing Width | Epoch 15/100 | Train Loss: 0.7020 | Validation Loss: 1.9467 | Validation Accuracy: 51.72%
Experiment 1 - Increasing Width | Epoch 20/100 | Train Loss: 0.5612 | Validation Loss: 2.2198 | Validation Accuracy: 52.36%
Experiment 1 - Increasing Width | Epoch 25/100 | Train Loss: 0.4694 | Validation Loss: 2.6161 | Validation Accuracy: 51.30%
Experiment 1 - Increasing Width | Epoch 30/100 | Train Loss: 0.4111 | Validation Loss: 2.9178 | Validation Accuracy: 51.16%
Experiment 1 - Increasing Width | Epoch 35/100 | Train Loss: 0.3733 | Validation Loss: 3.0859 | Validation Accuracy: 50.32%
Experiment 1 - Increasing Width | Epoch 40/100 | Train Loss: 0.3400 | Validation Loss: 3.3193 | Validation Accuracy: 52.08%
Experimen

In [11]:
results5 = train_model(model5, "Experiment 2 - Sigmoid Activation")

Experiment 2 - Sigmoid Activation | Epoch 5/100 | Train Loss: 1.4011 | Validation Loss: 1.4926 | Validation Accuracy: 47.32%
Experiment 2 - Sigmoid Activation | Epoch 10/100 | Train Loss: 1.1971 | Validation Loss: 1.4435 | Validation Accuracy: 50.22%
Experiment 2 - Sigmoid Activation | Epoch 15/100 | Train Loss: 1.0223 | Validation Loss: 1.4838 | Validation Accuracy: 49.56%
Experiment 2 - Sigmoid Activation | Epoch 20/100 | Train Loss: 0.8709 | Validation Loss: 1.5624 | Validation Accuracy: 50.36%
Experiment 2 - Sigmoid Activation | Epoch 25/100 | Train Loss: 0.7209 | Validation Loss: 1.6702 | Validation Accuracy: 48.78%
Experiment 2 - Sigmoid Activation | Epoch 30/100 | Train Loss: 0.5967 | Validation Loss: 1.8268 | Validation Accuracy: 47.96%
Experiment 2 - Sigmoid Activation | Epoch 35/100 | Train Loss: 0.5048 | Validation Loss: 1.9502 | Validation Accuracy: 48.16%
Experiment 2 - Sigmoid Activation | Epoch 40/100 | Train Loss: 0.4144 | Validation Loss: 2.0766 | Validation Accuracy: 

In [12]:
results6 = train_model(model6, "Experiment 3 - Tanh Activation")

Experiment 3 - Tanh Activation | Epoch 5/100 | Train Loss: 1.5447 | Validation Loss: 1.6269 | Validation Accuracy: 43.06%
Experiment 3 - Tanh Activation | Epoch 10/100 | Train Loss: 1.4147 | Validation Loss: 1.5696 | Validation Accuracy: 45.74%
Experiment 3 - Tanh Activation | Epoch 15/100 | Train Loss: 1.3040 | Validation Loss: 1.5524 | Validation Accuracy: 46.62%
Experiment 3 - Tanh Activation | Epoch 20/100 | Train Loss: 1.2292 | Validation Loss: 1.6021 | Validation Accuracy: 45.54%
Experiment 3 - Tanh Activation | Epoch 25/100 | Train Loss: 1.1607 | Validation Loss: 1.6073 | Validation Accuracy: 45.38%
Experiment 3 - Tanh Activation | Epoch 30/100 | Train Loss: 1.0866 | Validation Loss: 1.6516 | Validation Accuracy: 46.00%
Experiment 3 - Tanh Activation | Epoch 35/100 | Train Loss: 1.0367 | Validation Loss: 1.6894 | Validation Accuracy: 46.22%
Experiment 3 - Tanh Activation | Epoch 40/100 | Train Loss: 1.0163 | Validation Loss: 1.6939 | Validation Accuracy: 46.20%
Experiment 3 - Ta